# WikiRank en Databricks: de local a la nube

**Equipo:** Kinga Kerekes, Laura Sánchez, Raúl Pérez

Adaptación a Databricks del pipeline de la Entrega 1 (`pipeline.ipynb`), que se ejecutaba
en un clúster Docker con HDFS (2 DataNodes, replicación 2) y Spark (1 worker, 2 núcleos, 3 GB).

## Antes de ejecutar
1. Ejecutar en local `seleccion_datos.py` - genera `wikirank_subset.csv` (≤ 2 GB) y `wikirank_subset_metadata.json`.
2. Ejecutar las dos primeras celdas (crean el esquema y el volume `raw`) y subir los dos ficheros al volume:
   *Catalog -> workspace -> wikirank -> raw -> Upload to this volume*.
3. *Run all*. La celda de `%pip` va la primera porque en Databricks reinicia el intérprete de Python.

In [0]:
%pip install langcodes==3.5.1 language_data==1.4.0

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Configuración (equivalente a config.yaml)
import os, json, time
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType

CATALOG, SCHEMA, VOLUME = "workspace", "wikirank", "raw"
VOL = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
CSV_FILE = f"{VOL}/wikirank_subset.csv"
META_FILE = f"{VOL}/wikirank_subset_metadata.json"
T = lambda layer, name: f"{CATALOG}.{SCHEMA}.{layer}_{name}"  

cfg = {
    "partition_column": "Language",
    "quality": {"score_min": 0, "score_max": 100, "max_bad_ratio": 0.05},
}

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

TIMES = {}   
def timed(name, fn):
    t = time.time(); out = fn(); TIMES[name] = round(time.time() - t, 1)
    print(f"{name}: {TIMES[name]} s"); return out

def save(df, layer, name, partition=None):
    w = df.write.mode("overwrite").option("overwriteSchema", "true")
    if partition:
        w = w.partitionBy(partition)
    w.saveAsTable(T(layer, name))

print("Spark:", spark.version, "| DBR:", os.environ.get("DATABRICKS_RUNTIME_VERSION"))

Spark: 4.2.0 | DBR: client.6.5


## 0. Entorno

In [0]:
def try_conf(key, value):
    try:
        spark.conf.set(key, value)
        print(f"PERMITIDO     {key} = {value}")
    except Exception as e:
        print(f"NO PERMITIDO  {key}: {str(e).splitlines()[0][:110]}")

for k, v in [("spark.executor.memory", "3g"),
             ("spark.hadoop.dfs.replication", "2"),
             ("spark.sql.autoBroadcastJoinThreshold", "-1"),
             ("spark.sql.adaptive.enabled", "false"),
             ("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "3"),
             ("spark.sql.shuffle.partitions", "auto"),
             ("spark.sql.session.timeZone", "Europe/Madrid")]:
    try_conf(k, v)

for k in ["spark.sql.ansi.enabled", "spark.sql.adaptive.enabled", "spark.sql.shuffle.partitions",
          "spark.sql.autoBroadcastJoinThreshold", "spark.databricks.photon.enabled"]:
    try:
        print(f"{k:42s} {spark.conf.get(k)}")
    except Exception as e:
        print(f"{k:42s} (no legible)")

NO PERMITIDO  spark.executor.memory: [CONFIG_NOT_AVAILABLE.WITHOUT_SUGGESTION] Configuration spark.executor.memory is not available.  SQLSTATE: 42K
NO PERMITIDO  spark.hadoop.dfs.replication: [CONFIG_NOT_AVAILABLE.WITHOUT_SUGGESTION] Configuration spark.hadoop.dfs.replication is not available.  SQLSTA
NO PERMITIDO  spark.sql.autoBroadcastJoinThreshold: [CONFIG_NOT_AVAILABLE.WITHOUT_SUGGESTION] Configuration spark.sql.autoBroadcastJoinThreshold is not available.
NO PERMITIDO  spark.sql.adaptive.enabled: [CONFIG_NOT_AVAILABLE.WITHOUT_SUGGESTION] Configuration spark.sql.adaptive.enabled is not available.  SQLSTATE
NO PERMITIDO  spark.sql.adaptive.skewJoin.skewedPartitionFactor: [CONFIG_NOT_AVAILABLE.WITHOUT_SUGGESTION] Configuration spark.sql.adaptive.skewJoin.skewedPartitionFactor is n
PERMITIDO     spark.sql.shuffle.partitions = auto
PERMITIDO     spark.sql.session.timeZone = Europe/Madrid
spark.sql.ansi.enabled                     true
spark.sql.adaptive.enabled                 (no leg

## 1. Ingesta: capa raw (volume)

In [0]:
with open(META_FILE, encoding="utf-8") as fm:     
    meta = json.load(fm)
FRAC = meta["fraccion"]

info = {f.name: f.size for f in dbutils.fs.ls(VOL)}
size = info.get(os.path.basename(CSV_FILE))
assert size is not None, f"Falta {CSV_FILE}: súbelo al volume"
assert size == meta["bytes_seleccion"], f"CSV incompleto: {size} bytes, se esperaban {meta['bytes_seleccion']}"
print(f"CSV en el volume y completo: {size:,} bytes | filas: {meta['filas_seleccion']:,} "
      f"| fracción común: {FRAC:.4f} | mínimo por lengua: {meta['min_por_lengua']:,}")
display(dbutils.fs.ls(VOL))

CSV en el volume y completo: 1,890,380,375 bytes | filas: 10,909,814 | fracción común: 0.2320 | mínimo por lengua: 10,000


path,name,size,modificationTime
dbfs:/Volumes/workspace/wikirank/raw/wikirank_subset.csv,wikirank_subset.csv,1890380375,1795086482000
dbfs:/Volumes/workspace/wikirank/raw/wikirank_subset_metadata.json,wikirank_subset_metadata.json,10812,1795082917000


In [0]:
schema = StructType([
    StructField("Wikipedia_article_URL", StringType()),
    StructField("Language", StringType()),
    StructField("Page_ID", LongType()),
    StructField("Title", StringType()),
    StructField("WikiRank_score", DoubleType()),
    StructField("WikiRank_URL", StringType()),
])
raw_csv = (spark.read.schema(schema)
             .option("header", True)
             .option("escape", '"')
             .option("multiLine", False)
             .csv(CSV_FILE))
raw_csv.printSchema()
raw_csv.show(3, truncate=40)

root
 |-- Wikipedia_article_URL: string (nullable = true)
 |-- Language: string (nullable = true)
 |-- Page_ID: long (nullable = true)
 |-- Title: string (nullable = true)
 |-- WikiRank_score: double (nullable = true)
 |-- WikiRank_URL: string (nullable = true)

+----------------------------------------+--------+-------+------+--------------+----------------------------------------+
|                   Wikipedia_article_URL|Language|Page_ID| Title|WikiRank_score|                            WikiRank_URL|
+----------------------------------------+--------+-------+------+--------------+----------------------------------------+
|         https://ar.wikipedia.org/wiki/A|      ar|     77|     A|       21.2151|               https://wikirank.net/ar/A|
|         https://ar.wikipedia.org/wiki/K|      ar|     80|     K|       14.2644|               https://wikirank.net/ar/K|
|https://ar.wikipedia.org/wiki/%D8%B9%...|      ar|     90|عنكبوت|       18.2658|https://wikirank.net/ar/%D8%B9%D9%86%...|

### 1.1 Particionado de la lectura


In [0]:
parts = (raw_csv.groupBy(F.spark_partition_id().alias("particion")).count()
            .agg(F.count("*").alias("n_particiones"), F.min("count").alias("min_filas"),
                 F.max("count").alias("max_filas"), F.round(F.avg("count")).alias("media_filas")))
parts.show()
try:
    print("maxPartitionBytes:", spark.conf.get("spark.sql.files.maxPartitionBytes"))
except Exception:
    pass

+-------------+---------+---------+-----------+
|n_particiones|min_filas|max_filas|media_filas|
+-------------+---------+---------+-----------+
|           15|    42383|   997072|   727321.0|
+-------------+---------+---------+-----------+

maxPartitionBytes: 134217728b


### 1.2 Capa bronze

In [0]:
timed("bronze", lambda: save(raw_csv.withColumn("_source_file", F.col("_metadata.file_path"))
                                    .withColumn("_ingested_at", F.current_timestamp()),
                             "bronze", "articles"))
bronze = spark.table(T("bronze", "articles"))
n_bronze = bronze.count()
assert n_bronze == meta["filas_seleccion"], f"bronze: {n_bronze:,} filas, se esperaban {meta['filas_seleccion']:,}"
print(f"bronze_articles: {n_bronze:,} filas (= filas del subconjunto)")

raw = bronze.drop("_source_file", "_ingested_at")   # las 6 columnas originales, para la capa silver

bronze: 8.9 s
bronze_articles: 10,909,814 filas (= filas del subconjunto)


## 2. Capa silver: controles de calidad y cuarentena

In [0]:
q = cfg["quality"]
rule = (F.col("Language").isNotNull() & F.col("Page_ID").isNotNull()
        & F.col("WikiRank_score").between(q["score_min"], q["score_max"]))
ok = F.coalesce(rule, F.lit(False))

def quality():
    c = raw.agg(
        F.count("*").alias("filas_total"),
        *[F.sum(F.col(x).isNull().cast("int")).alias("nulos_" + x) for x in raw.columns],
        F.sum((~F.col("WikiRank_score").between(q["score_min"], q["score_max"])).cast("int")).alias("score_fuera_de_rango"),
        F.sum((~ok).cast("int")).alias("filas_invalidas"),
    ).first().asDict()
    c["claves_duplicadas"] = (raw.groupBy(F.lower(F.trim("Language")).alias("Language"), "Page_ID")
                                 .count().filter("count > 1").count())
    return {k: (v or 0) for k, v in c.items()}

checks = timed("calidad", quality)
for k, v in checks.items():
    print(f"{k:28s} {v}")

calidad: 2.3 s
filas_total                  10909814
nulos_Wikipedia_article_URL  0
nulos_Language               0
nulos_Page_ID                0
nulos_Title                  0
nulos_WikiRank_score         0
nulos_WikiRank_URL           0
score_fuera_de_rango         0
filas_invalidas              0
claves_duplicadas            0


In [0]:
good, bad = raw.filter(ok), raw.filter(~ok)
save(bad, "silver", "quarantine_articles")
n_bad = checks["filas_invalidas"]
print("Filas en cuarentena:", n_bad)
assert n_bad / checks["filas_total"] < q["max_bad_ratio"], f"demasiadas filas malas: {n_bad}"

Filas en cuarentena: 0


## 3. Capa silver: artículos limpios, particionados por lengua

In [0]:
FRAC = meta["fraccion"]
fractions = spark.createDataFrame(
    [(k, float(v["fraccion_lengua"]), v["n_original"], v["media_original"], v["n_seleccion"], v["top10_forzadas"])
     for k, v in meta["lenguas"].items()],
    "Language STRING, f_lang DOUBLE, n_original LONG, media_original DOUBLE, n_seleccion LONG, top10_forzadas LONG")
save(fractions, "silver", "sampling_fractions")

url = F.col("Wikipedia_article_URL")
u_expr = F.crc32(F.encode(F.concat_ws("|", F.lit(meta["semilla_hash"]), "Language",
                                      F.col("Page_ID").cast("string")), "UTF-8")) / F.lit(4294967296.0)
curated = (good
    .withColumn("Language", F.lower(F.trim("Language")))
    .withColumn("Title", F.coalesce(F.col("Title"), F.lit("")))
    .withColumn("url_pct_count", (F.length(url) - F.length(F.regexp_replace(url, "%", ""))).cast("int"))
    .drop("Wikipedia_article_URL", "WikiRank_URL")
    .join(F.broadcast(spark.table(T("silver", "sampling_fractions")).select("Language", "f_lang")), "Language", "left")
    .withColumn("u", u_expr)
    .withColumn("en_muestra", F.col("u") <= F.col("f_lang"))
    .withColumn("en_muestra_comun", F.col("u") <= F.lit(FRAC)))
if checks["claves_duplicadas"] > 0:
    curated = curated.dropDuplicates(["Language", "Page_ID"])

part = cfg["partition_column"]
timed("escritura_curada", lambda: save(curated.repartition(part), "silver", "articles", partition=part))

ARTICLES = T("silver", "articles")
articles = spark.table(ARTICLES)
checks["filas_curadas"] = articles.count()
checks["filas_eliminadas"] = checks["filas_total"] - checks["filas_curadas"]
print("Filas curadas:", checks["filas_curadas"], "| eliminadas:", checks["filas_eliminadas"])

escritura_curada: 11.6 s
Filas curadas: 10909814 | eliminadas: 0


In [0]:
# Comprobación del hash: toda fila que no salió por el muestreo tiene que estar en el top 10 de su lengua.
# Si da 0, el u de Spark es idéntico al de seleccion_datos.py.
display(spark.sql(f"""
    SELECT COUNT(*)                                        AS filas,
           COUNT_IF(en_muestra)                            AS en_muestra,
           COUNT_IF(NOT en_muestra)                        AS solo_por_top10,
           COUNT_IF(NOT en_muestra AND rk > 10)            AS fuera_de_muestra_y_top10
    FROM (SELECT en_muestra,
                 ROW_NUMBER() OVER (PARTITION BY Language ORDER BY WikiRank_score DESC, Page_ID ASC) AS rk
          FROM {ARTICLES})
"""))

filas,en_muestra,solo_por_top10,fuera_de_muestra_y_top10
10909814,10909383,431,0


In [0]:
quality_df = spark.createDataFrame([checks])
save(quality_df, "gold", "quality_report")
display(quality_df)

claves_duplicadas,filas_curadas,filas_eliminadas,filas_invalidas,filas_total,nulos_Language,nulos_Page_ID,nulos_Title,nulos_WikiRank_URL,nulos_WikiRank_score,nulos_Wikipedia_article_URL,score_fuera_de_rango
0,10909814,0,0,10909814,0,0,0,0,0,0,0


In [0]:
# Ficheros físicos de la tabla Delta (equivale a `hdfs dfs -du` / fsck de la zona curada)
display(spark.sql(f"DESCRIBE DETAIL {ARTICLES}")
        .select("format", "partitionColumns", "clusteringColumns", "numFiles", "sizeInBytes", "location"))

# Filas por fichero: muestra el desequilibrio (skew) de la escritura por lengua
files = (articles.groupBy("Language", F.col("_metadata.file_path").alias("fichero")).count()
                 .orderBy(F.desc("count")))
files.show(5, truncate=60)

format,partitionColumns,clusteringColumns,numFiles,sizeInBytes,location
delta,List(Language),List(),55,215998483,


+--------+------------------------------------------------------------+-------+
|Language|                                                     fichero|  count|
+--------+------------------------------------------------------------+-------+
|      en|s3://dbstorage-prod-p9wbfadnx8/uc/9ab49f53-b857-482e-9b5a...|1592301|
|      de|s3://dbstorage-prod-p9wbfadnx8/uc/9ab49f53-b857-482e-9b5a...| 679550|
|      fr|s3://dbstorage-prod-p9wbfadnx8/uc/9ab49f53-b857-482e-9b5a...| 609096|
|      sv|s3://dbstorage-prod-p9wbfadnx8/uc/9ab49f53-b857-482e-9b5a...| 601386|
|      nl|s3://dbstorage-prod-p9wbfadnx8/uc/9ab49f53-b857-482e-9b5a...| 501528|
+--------+------------------------------------------------------------+-------+
only showing top 5 rows


In [0]:
# Partition pruning: el filtro por lengua debe aparecer como PartitionFilters
es = spark.sql(f"SELECT * FROM {ARTICLES} WHERE Language = 'es'")
es.explain("formatted")
print("Artículos en es:", es.count())

== Physical Plan ==
PhotonResultStage (4)
+- PhotonColumnarToRow (3)
   +- PhotonProject (2)
      +- PhotonScan parquet workspace.wikirank.silver_articles (1)


(1) PhotonScan parquet workspace.wikirank.silver_articles
Output [9]: [Page_ID#26262L, Title#26263, WikiRank_score#26264, url_pct_count#26265, f_lang#26266, u#26267, en_muestra#26268, en_muestra_comun#26269, Language#26261]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-p9wbfadnx8/uc/9ab49f53-b857-482e-9b5a-5c2cd49c0413/9760e452-f375-4333-aee3-53e5704b0f6d/__unitystorage/catalogs/e3a48b33-20f9-4567-89ef-90a8e1f9e796/tables/66ecf638-2c7b-44d5-84aa-5edd62012521]
PartitionFilters: [isnotnull(Language#26261), (Language#26261 = es)]
ReadSchema: struct<Page_ID:bigint,Title:string,WikiRank_score:double,url_pct_count:int,f_lang:double,u:double,en_muestra:boolean,en_muestra_comun:boolean>

(2) PhotonProject
Input [9]: [Page_ID#26262L, Title#26263, WikiRank_score#26264, url_pct_count#26265, f_lang#26266, u#26267, en_muestra#26268

## 4. Capa silver: tabla auxiliar de idiomas (`language_info`)

In [0]:
import langcodes

CODE_ALIAS = {"simple": "en"}
FIX_NAME = {"sh": "Serbo-Croatian", "simple": "Simple English"}
FIX_SCRIPT = {"sh": "Latn", "zh-min-nan": "Latn"}

FAMILY = {"ar": "Afro-Asiatic", "he": "Afro-Asiatic", "az": "Turkic", "kk": "Turkic", "tr": "Turkic",
          "uz": "Turkic", "et": "Uralic", "fi": "Uralic", "hu": "Uralic", "eu": "Isolate",
          "ka": "Kartvelian", "ja": "Japonic", "ko": "Koreanic", "id": "Austronesian",
          "ms": "Austronesian", "ta": "Dravidian", "th": "Kra-Dai", "vi": "Austroasiatic",
          "zh": "Sino-Tibetan", "zh-min-nan": "Sino-Tibetan", "eo": "Constructed", "vo": "Constructed"}
for c in ["be", "bg", "ca", "cs", "da", "de", "el", "en", "es", "fa", "fr", "gl", "hi", "hr", "hy", "it",
          "la", "lt", "nl", "nn", "no", "pl", "pt", "ro", "ru", "sh", "simple", "sk", "sl", "sr", "sv", "uk", "ur"]:
    FAMILY[c] = "Indo-European"

FSI = {"da": 1, "es": 1, "fr": 1, "it": 1, "nl": 1, "nn": 1, "no": 1, "pt": 1, "ro": 1, "sv": 1,
       "de": 2, "id": 2, "ms": 2,
       "az": 3, "be": 3, "bg": 3, "cs": 3, "el": 3, "et": 3, "fa": 3, "fi": 3, "he": 3, "hi": 3, "hr": 3,
       "hu": 3, "hy": 3, "ka": 3, "kk": 3, "lt": 3, "pl": 3, "ru": 3, "sh": 3, "sk": 3, "sl": 3, "sr": 3,
       "ta": 3, "th": 3, "tr": 3, "uk": 3, "ur": 3, "uz": 3, "vi": 3,
       "ar": 4, "ja": 4, "ko": 4, "zh": 4}

def lang_row(code):
    lang = langcodes.Language.get(CODE_ALIAS.get(code, code))
    return (code, FIX_NAME.get(code, lang.display_name("en")),
            round(lang.speaking_population() / 1e6, 3),
            FIX_SCRIPT.get(code, lang.maximize().script), FAMILY.get(code), FSI.get(code))

codes = sorted(r["Language"] for r in articles.select("Language").distinct().collect())
lang_info = spark.createDataFrame([lang_row(c) for c in codes],
    "Language STRING, language_name STRING, speakers_millions DOUBLE, script STRING, family STRING, fsi_category INT")

dups = lang_info.groupBy("Language").count().filter("count > 1").count()
assert dups == 0, f"{dups} lenguas duplicadas en language_info"
lang_info.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(T("silver", "language_info"))
lang_info = spark.table(T("silver", "language_info"))
print(f"language_info: {lang_info.count()} lenguas, sin duplicados")

language_info: 55 lenguas, sin duplicados


## 5. Capa gold: preguntas de negocio en Spark SQL


**P1. Artículos y media por idioma.** 

In [0]:
LANG_INFO = T("silver", "language_info")
FRACTIONS = T("silver", "sampling_fractions")

q1_sql = spark.sql(f"""
    SELECT Language,
           COUNT_IF(en_muestra)                                        AS n_sample,
           CAST(ROUND(COUNT_IF(en_muestra) / MAX(f_lang)) AS BIGINT)   AS n_articles,
           ROUND(AVG(CASE WHEN en_muestra THEN WikiRank_score END), 2) AS avg_score
    FROM {ARTICLES}
    GROUP BY Language
    ORDER BY n_articles DESC
""")
timed("P1", lambda: save(q1_sql, "gold", "q1_stats_by_language"))
Q1 = T("gold", "q1_stats_by_language")
display(spark.table(Q1))

P1: 2.6 s


Language,n_sample,n_articles,avg_score
en,1592293,6863386,24.71
de,679540,2929075,18.65
fr,609088,2625400,16.11
sv,601379,2592171,20.04
nl,501520,2161741,12.71
ru,462230,1992386,18.2
es,441214,1901799,18.8
it,435447,1876941,13.6
pl,376212,1621616,17.04
zh,331891,1430576,20.29


**Comparativa P1 con el clúster local.** 

In [0]:
LOCAL_P1 = """ar 1237943 17.109;az 201054 15.343;be 244985 12.946;bg 299787 17.745;ca 756348 21.161;cs 550666 16.441;da 301275 9.778;de 2930971 18.632;el 237671 23.000;en 6860430 24.712;eo 356698 11.676;es 1905093 18.765;et 246512 18.069;eu 430677 15.759;fa 1008969 11.428;fi 577535 17.645;fr 2626210 16.118;gl 212505 14.166;he 359029 21.801;hi 166951 16.178;hr 207360 20.155;hu 544427 19.037;hy 312200 18.046;id 696238 13.929;it 1875385 13.628;ja 1424380 14.567;ka 173955 13.224;kk 239916 13.072;ko 679238 10.438;la 139220 16.815;lt 215369 13.739;ms 387480 14.387;nl 2163715 12.728;nn 170159 18.870;no 633769 15.529;pl 1624175 17.028;pt 1130161 17.710;ro 464908 16.614;ru 1991787 18.182;sh 459884 18.657;simple 254977 17.480;sk 249617 17.455;sl 186122 17.872;sr 692273 15.904;sv 2591505 20.019;ta 168458 25.991;th 165837 18.483;tr 617404 18.135;uk 1339146 18.817;ur 211138 13.982;uz 299908 14.608;vi 1294722 14.244;vo 37495 16.260;zh 1433026 20.309;zh-min-nan 433220 21.541"""
local = spark.createDataFrame([(l, int(n), float(a)) for l, n, a in (x.split() for x in LOCAL_P1.split(";"))],
                              "Language STRING, n_local LONG, avg_local DOUBLE")
local.createOrReplaceTempView("local_p1")

cmp = spark.sql(f"""
    SELECT COALESCE(q.Language, l.Language)                              AS Language,
           q.n_articles, l.n_local,
           ROUND(100 * (q.n_articles - l.n_local) / l.n_local, 3)      AS err_n_pct,
           q.avg_score, l.avg_local,
           ROUND(q.avg_score - l.avg_local, 3)                          AS dif_media,
           f.n_original
    FROM {Q1} q
    FULL OUTER JOIN local_p1 l ON q.Language = l.Language
    LEFT JOIN {FRACTIONS} f    ON q.Language = f.Language
    ORDER BY l.n_local DESC
""")
cmp.createOrReplaceTempView("cmp_p1")
display(cmp)
display(spark.sql("""
    SELECT COUNT(*) AS lenguas, SUM(n_local) AS total_local, SUM(n_articles) AS total_estimado,
           MAX(ABS(err_n_pct)) AS max_err_n_pct, MAX(ABS(dif_media)) AS max_dif_media,
           COUNT_IF(n_local <> n_original) AS lenguas_con_n_distinto_script_vs_local
    FROM cmp_p1
"""))

Language,n_articles,n_local,err_n_pct,avg_score,avg_local,dif_media,n_original
en,6863386,6860430,0.043,24.71,24.712,-0.002,6860430
de,2929075,2930971,-0.065,18.65,18.632,0.018,2930971
fr,2625400,2626210,-0.031,16.11,16.118,-0.008,2626210
sv,2592171,2591505,0.026,20.04,20.019,0.021,2591505
nl,2161741,2163715,-0.091,12.71,12.728,-0.018,2163715
ru,1992386,1991787,0.03,18.2,18.182,0.018,1991787
es,1901799,1905093,-0.173,18.8,18.765,0.035,1905093
it,1876941,1875385,0.083,13.6,13.628,-0.028,1875385
pl,1621616,1624175,-0.158,17.04,17.028,0.012,1624175
zh,1430576,1433026,-0.171,20.29,20.309,-0.019,1433026


lenguas,total_local,total_estimado,max_err_n_pct,max_dif_media,lenguas_con_n_distinto_script_vs_local
55,47019883,47018319,0.949,0.138,0


**P2. Top 10 de cada idioma**

In [0]:
q2_sql = spark.sql(f"""
    SELECT Language, rank, Title, WikiRank_score, Page_ID
    FROM (SELECT Language, Title, WikiRank_score, Page_ID,
                 ROW_NUMBER() OVER (PARTITION BY Language ORDER BY WikiRank_score DESC, Page_ID ASC) AS rank
          FROM {ARTICLES})
    WHERE rank <= 10
""")
timed("P2", lambda: save(q2_sql, "gold", "q2_top10_by_language"))
Q2 = T("gold", "q2_top10_by_language")
display(spark.sql(f"SELECT * FROM {Q2} WHERE Language = 'es' ORDER BY rank"))

P2: 2.9 s


Language,rank,Title,WikiRank_score,Page_ID
es,1,Apolo,100.0,293
es,2,Budismo,100.0,317
es,3,Brasil,100.0,410
es,4,Camino de Santiago,100.0,765
es,5,Heráclito,100.0,1429
es,6,Imperio romano,100.0,1509
es,7,Japón,100.0,1575
es,8,Johann Sebastian Bach,100.0,1591
es,9,Orchidaceae,100.0,2062
es,10,Pablo Picasso,100.0,2147


**P3a. Hablantes frente a tamaño y calidad**

In [0]:
q3a_sql = spark.sql(f"""
    SELECT q.Language, i.language_name, i.speakers_millions, q.n_articles,
           CAST(ROUND(q.n_articles / i.speakers_millions) AS BIGINT) AS articles_per_million,
           q.avg_score,
           LOG10(i.speakers_millions) AS log_speakers,
           LOG10(q.n_articles)        AS log_articles
    FROM {Q1} q
    JOIN {LANG_INFO} i ON q.Language = i.Language
    WHERE q.Language NOT IN ('simple', 'sh', 'la', 'eo', 'vo')
""")
timed("P3a", lambda: save(q3a_sql, "gold", "q3_speakers_by_language"))
Q3A = T("gold", "q3_speakers_by_language")
display(spark.sql(f"""
    SELECT Language, language_name, speakers_millions, n_articles, articles_per_million, avg_score
    FROM {Q3A} ORDER BY articles_per_million DESC
"""))

corr_sql = spark.sql(f"""
    SELECT COUNT(*)                                          AS n_languages,
           ROUND(COVAR_SAMP(log_speakers, log_articles), 3)  AS covar_speakers_articles,
           ROUND(CORR(log_speakers, log_articles), 3)        AS corr_speakers_articles,
           ROUND(CORR(log_speakers, avg_score), 3)           AS corr_speakers_score
    FROM {Q3A}
""")
save(corr_sql, "gold", "q3_speakers")
display(spark.table(T("gold", "q3_speakers")))

P3a: 2.4 s


Language,language_name,speakers_millions,n_articles,articles_per_million,avg_score
eu,Basque,1.034,431357,417173,15.78
et,Estonian,0.854,247855,290228,18.03
sv,Swedish,13.338,2592171,194345,20.04
nn,Norwegian Nynorsk,1.377,170782,124025,18.87
no,Norwegian,5.51,634272,115113,15.5
fi,Finnish,5.789,578285,99894,17.68
sl,Slovenian,1.967,186945,95041,18.01
ca,Catalan,8.219,755661,91941,21.14
lt,Lithuanian,2.405,216562,90047,13.69
nl,Dutch,32.855,2161741,65796,12.71


n_languages,covar_speakers_articles,corr_speakers_articles,corr_speakers_score
50,0.189,0.566,0.105


**P3b. Distribución por escritura, familia y categoría FSI**

In [0]:
q3_traits = spark.sql(f"""
    SELECT dimension, value,
           COUNT(DISTINCT Language)                                       AS n_languages,
           CAST(ROUND(COUNT(*) / {FRAC}) AS BIGINT)                       AS n_articles_est,
           ROUND(AVG(WikiRank_score), 2)                                  AS avg_score,
           PERCENTILE_APPROX(WikiRank_score, 0.5, 10000)                  AS median_score,
           PERCENTILE_APPROX(WikiRank_score, 0.9, 10000)                  AS p90_score,
           ROUND(AVG(CAST(WikiRank_score >= 50 AS INT)) * 100, 1)         AS pct_score_50
    FROM (SELECT /*+ BROADCAST(i) */ a.Language, a.WikiRank_score,
                 STACK(3, 'script', i.script,
                          'family', i.family,
                          'fsi_category', COALESCE(CAST(i.fsi_category AS STRING), 'sin clasificar')) AS (dimension, value)
          FROM {ARTICLES} a
          LEFT JOIN {LANG_INFO} i ON a.Language = i.Language
          WHERE a.en_muestra_comun)
    GROUP BY dimension, value
""")
timed("P3b", lambda: save(q3_traits, "gold", "q3_by_language_traits"))
Q3B = T("gold", "q3_by_language_traits")
for d in ["script", "family", "fsi_category"]:
    display(spark.sql(f"SELECT * FROM {Q3B} WHERE dimension = '{d}' ORDER BY n_articles_est DESC"))

P3b: 3.4 s


dimension,value,n_languages,n_articles_est,avg_score,median_score,p90_score,pct_score_50
script,Latn,36,34630255,18.34,17.0272,34.1511,3.0
script,Cyrl,6,4811486,17.5,17.17,30.8135,1.5
script,Arab,3,2452903,14.5,13.7343,25.4507,0.6
script,Hans,1,1430576,20.29,17.8248,40.6162,5.3
script,Jpan,1,1427145,14.55,11.78,29.5209,1.7
script,Kore,1,679872,10.46,6.5833,25.0231,0.9
script,Hebr,1,359874,21.88,20.0336,41.0982,4.8
script,Armn,1,311774,18.09,19.441,29.971,1.2
script,Grek,1,238286,23.05,24.0459,39.317,4.6
script,Geor,1,173971,13.3,8.2117,28.9138,1.1


dimension,value,n_languages,n_articles_est,avg_score,median_score,p90_score,pct_score_50
family,Indo-European,33,35008116,18.46,17.5745,33.7189,3.0
family,Sino-Tibetan,2,1863515,20.58,18.2601,39.5459,4.1
family,Afro-Asiatic,2,1595801,18.18,17.9039,30.0136,1.5
family,Japonic,1,1427145,14.55,11.78,29.5209,1.7
family,Uralic,3,1372748,18.3,17.4617,32.1927,2.3
family,Turkic,4,1359282,16.06,14.5089,28.9885,1.4
family,Austroasiatic,1,1293359,14.22,12.4114,23.1332,0.8
family,Austronesian,2,1084198,14.11,9.5944,32.0831,3.0
family,Koreanic,1,679872,10.46,6.5833,25.0231,0.9
family,Isolate,1,431357,15.78,12.7098,32.9192,0.9


dimension,value,n_languages,n_articles_est,avg_score,median_score,p90_score,pct_score_50
fsi_category,3,29,14893716,16.95,15.4601,31.1703,1.7
fsi_category,1,10,13852656,16.36,16.3834,28.7909,1.5
fsi_category,sin clasificar,9,9485070,22.8,20.9254,42.2229,6.1
fsi_category,4,4,4773520,16.35,14.4417,30.9719,2.4
fsi_category,2,3,4013273,17.43,14.6328,35.588,3.3


## 6. Planes de ejecución

In [0]:
# P1
q1_sql.explain("extended")

== Parsed Logical Plan ==
'Sort ['n_articles DESC NULLS LAST], true
+- 'Aggregate ['Language], ['Language, 'COUNT_IF('en_muestra) AS n_sample#27579, cast('ROUND(('COUNT_IF('en_muestra) / 'MAX('f_lang))) as bigint) AS n_articles#27580, 'ROUND('AVG(CASE WHEN 'en_muestra THEN 'WikiRank_score END), 2) AS avg_score#27581]
   +- 'UnresolvedRelation [workspace, wikirank, silver_articles], [], false

== Analyzed Logical Plan ==
Language: string, n_sample: bigint, n_articles: bigint, avg_score: double
Sort [n_articles#27580L DESC NULLS LAST], true
+- Aggregate [Language#31176], [Language#31176, count_if(en_muestra#31183) AS n_sample#27579L, cast(round((cast(count_if(en_muestra#31183) as double) / cast(max(f_lang#31181) as double)), 0) as bigint) AS n_articles#27580L, round(avg(CASE WHEN en_muestra#31183 THEN WikiRank_score#31179 END), 2) AS avg_score#27581]
   +- SubqueryAlias workspace.wikirank.silver_articles
      +- Relation workspace.wikirank.silver_articles[Language#31176,Page_ID#31177L,T

In [0]:
q1_sql.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (14)
+- == Initial Plan ==
   PhotonResultStage (13)
   +- PhotonColumnarToRow (12)
      +- PhotonSort (11)
         +- PhotonShuffleExchangeSource (10)
            +- PhotonShuffleMapStage (9)
               +- PhotonShuffleExchangeSink (8)
                  +- PhotonGroupingAgg (7)
                     +- PhotonShuffleExchangeSource (6)
                        +- PhotonShuffleMapStage (5)
                           +- PhotonShuffleExchangeSink (4)
                              +- PhotonGroupingAgg (3)
                                 +- PhotonProject (2)
                                    +- PhotonScan parquet workspace.wikirank.silver_articles (1)


(1) PhotonScan parquet workspace.wikirank.silver_articles
Output [4]: [WikiRank_score#31262, f_lang#31264, en_muestra#31266, Language#31259]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-p9wbfadnx8/uc/9ab49f53-b857-482e-9b5a-5c2cd49c0413/9760e452-f375-4333-aee3-53e5704b0f6d/__unitystorage/c

In [0]:
# P2
q2_sql.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (16)
+- == Initial Plan ==
   PhotonResultStage (15)
   +- PhotonColumnarToRow (14)
      +- PhotonProject (13)
         +- PhotonFilter (12)
            +- PhotonWindow (11)
               +- PhotonTopK (10)
                  +- PhotonShuffleExchangeSource (9)
                     +- PhotonShuffleMapStage (8)
                        +- PhotonShuffleExchangeSink (7)
                           +- PhotonProject (6)
                              +- PhotonFilter (5)
                                 +- PhotonWindow (4)
                                    +- PhotonTopK (3)
                                       +- PhotonProject (2)
                                          +- PhotonScan parquet workspace.wikirank.silver_articles (1)


(1) PhotonScan parquet workspace.wikirank.silver_articles
Output [4]: [Page_ID#31339L, Title#31340, WikiRank_score#31341, Language#31338]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-p9wbfadnx8/uc/9ab49f53-b857-482

In [0]:
# P3b
q3_traits.explain("extended")

== Parsed Logical Plan ==
'Aggregate ['dimension, 'value], ['dimension, 'value, 'COUNT(distinct 'Language) AS n_languages#31100, cast('ROUND(('COUNT(1) / 0.2319981751254454)) as bigint) AS n_articles_est#31101, 'ROUND('AVG('WikiRank_score), 2) AS avg_score#31102, 'PERCENTILE_APPROX('WikiRank_score, 0.5, 10000) AS median_score#31103, 'PERCENTILE_APPROX('WikiRank_score, 0.9, 10000) AS p90_score#31104, 'ROUND(('AVG(cast(('WikiRank_score >= 50) as int)) * 100), 1) AS pct_score_50#31105]
+- 'SubqueryAlias __auto_generated_subquery_name
   +- 'UnresolvedHint BROADCAST, ['i]
      +- 'Project ['a.Language, 'a.WikiRank_score, 'STACK(3, script, 'i.script, family, 'i.family, fsi_category, 'COALESCE(cast('i.fsi_category as string), sin clasificar)) AS (dimension, value)]
         +- 'Filter 'a.en_muestra_comun
            +- 'Join LeftOuter, ('a.Language = 'i.Language)
               :- 'SubqueryAlias a
               :  +- 'UnresolvedRelation [workspace, wikirank, silver_articles], [], false
   

In [0]:
q3_traits.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (22)
+- == Initial Plan ==
   PhotonResultStage (21)
   +- PhotonColumnarToRow (20)
      +- PhotonGroupingAgg (19)
         +- PhotonShuffleExchangeSource (18)
            +- PhotonShuffleMapStage (17)
               +- PhotonShuffleExchangeSink (16)
                  +- PhotonGroupingAgg (15)
                     +- PhotonGroupingAgg (14)
                        +- PhotonShuffleExchangeSource (13)
                           +- PhotonShuffleMapStage (12)
                              +- PhotonShuffleExchangeSink (11)
                                 +- PhotonGroupingAgg (10)
                                    +- PhotonGenerate (9)
                                       +- PhotonProject (8)
                                          +- PhotonBroadcastHashJoin LeftOuter (7)
                                             :- PhotonProject (2)
                                             :  +- PhotonScan parquet workspace.wikirank.silver_articles (1)
   

## 7. Delta Lake: historial y versiones

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {ARTICLES}")
        .select("version", "timestamp", "operation", "operationParameters", "operationMetrics"))

version,timestamp,operation,operationParameters,operationMetrics
1,2026-11-20T09:27:50.343+01:00,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [""Language""], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""66ecf638-2c7b-44d5-84aa-5edd62012521"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-9776b437-5a65-48eb-a8b4-9dfdeea28de8"",""delta.enableInCommitTimestamps"":""true"",""databricks.internal.autoUpgrades.delta.enableInCommitTimestamps"":""true"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-6b458911-62b7-48e5-a2f1-5fccd7e3e0f4""}, statsOnLoad -> true)","Map(numFiles -> 55, numRemovedFiles -> 55, numRemovedBytes -> 215998483, numDeletionVectorsRemoved -> 0, numOutputRows -> 10909814, numOutputBytes -> 215998483)"
0,2026-11-20T09:20:47.163+01:00,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [""Language""], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""66ecf638-2c7b-44d5-84aa-5edd62012521"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-9776b437-5a65-48eb-a8b4-9dfdeea28de8"",""delta.enableInCommitTimestamps"":""true"",""databricks.internal.autoUpgrades.delta.enableInCommitTimestamps"":""true"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-6b458911-62b7-48e5-a2f1-5fccd7e3e0f4""}, statsOnLoad -> true)","Map(numFiles -> 55, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 10909814, numOutputBytes -> 215998483)"


In [0]:
hist = spark.sql(f"DESCRIBE HISTORY {ARTICLES}").select("version").collect()
v_min, v_max = min(r[0] for r in hist), max(r[0] for r in hist)
n_old = spark.read.option("versionAsOf", v_min).table(ARTICLES).count()
n_new = spark.table(ARTICLES).count()
print(f"Versión {v_min}: {n_old:,} filas | versión {v_max} (actual): {n_new:,} filas")
spark.sql(f"SELECT COUNT(*) AS filas_v{v_min} FROM {ARTICLES} VERSION AS OF {v_min}").show()

Versión 0: 10,909,814 filas | versión 1 (actual): 10,909,814 filas
+--------+
|filas_v0|
+--------+
|10909814|
+--------+



In [0]:
for layer, name in [("bronze", "articles"), ("silver", "articles"), ("gold", "q1_stats_by_language")]:
    h = spark.sql(f"DESCRIBE HISTORY {T(layer, name)}")
    print(f"{T(layer, name):45s} versiones: {h.count()} | última: {h.agg(F.max('version')).first()[0]}")

display(spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}"))  

workspace.wikirank.bronze_articles            versiones: 2 | última: 1
workspace.wikirank.silver_articles            versiones: 2 | última: 1
workspace.wikirank.gold_q1_stats_by_language  versiones: 2 | última: 1


database,tableName,isTemporary
wikirank,bronze_articles,false
wikirank,gold_q1_stats_by_language,false
wikirank,gold_q2_top10_by_language,false
wikirank,gold_q3_by_language_traits,false
wikirank,gold_q3_speakers,false
wikirank,gold_q3_speakers_by_language,false
wikirank,gold_quality_report,false
wikirank,gold_timings,false
wikirank,silver_articles,false
wikirank,silver_language_info,false


## 8. Resumen de tiempos

In [0]:
TIMES_LOCAL = {"calidad": 122, "escritura_curada": 118, "P1": 2.3, "P2": 42.0}
rows = [(k, float(v), float(TIMES_LOCAL[k]) if k in TIMES_LOCAL else None) for k, v in TIMES.items()]
times_df = spark.createDataFrame(rows, "etapa STRING, databricks_s DOUBLE, local_s DOUBLE")
save(times_df, "gold", "timings")
display(times_df)
print("Nota: los tiempos locales son sobre el dataset completo (8,2 GB); los de Databricks sobre el subconjunto.")

etapa,databricks_s,local_s
bronze,8.9,null
calidad,2.3,122.0
escritura_curada,11.6,118.0
P1,2.6,2.3
P2,2.9,42.0
P3a,2.4,null
P3b,3.4,null


Nota: los tiempos locales son sobre el dataset completo (8,2 GB); los de Databricks sobre el subconjunto.
